# Final Recommendation Map and Portfolio Tables

## Goal

Assemble the validated planning outputs into a final recommended-network map and compact portfolio tables. This notebook does not change the suitability or port-demand models; it only presents their results.

In [ ]:
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import pandas as pd

PROJECT_ROOT = Path(r"C:\Users\cason\GIS_Portfolio_3\Project 4 - EV Charging Suitability")
SPATIAL_DIR = PROJECT_ROOT / "outputs" / "spatial"
TABLE_DIR = PROJECT_ROOT / "outputs" / "tables"
MAP_DIR = PROJECT_ROOT / "maps"
FIGURE_DIR = PROJECT_ROOT / "outputs" / "figures"
for folder in [SPATIAL_DIR, TABLE_DIR, MAP_DIR, FIGURE_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

recommendation_path = SPATIAL_DIR / "recommended_ev_sites_and_equipment.gpkg"
study_area_path = SPATIAL_DIR / "study_area_counties.gpkg"
station_path = SPATIAL_DIR / "existing_ev_charging_units.gpkg"
interstate_path = PROJECT_ROOT / "data" / "raw" / "roads" / "gdot_interstates_study_area.gpkg"
scenario_path = TABLE_DIR / "ev_port_demand_scenarios.csv"

required = [recommendation_path, study_area_path, station_path, interstate_path, scenario_path]
missing = [path.name for path in required if not path.exists()]
if missing:
    raise FileNotFoundError(f"Run notebooks 02–09 first. Missing outputs: {missing}")

recommendations = gpd.read_file(recommendation_path, layer="recommendations").to_crs("EPSG:4326")
study_area = gpd.read_file(study_area_path, layer="study_counties").to_crs("EPSG:4326")
existing_units = gpd.read_file(station_path, layer="charging_units").to_crs("EPSG:4326")
interstates = gpd.read_file(interstate_path, layer="interstates").to_crs("EPSG:4326")
scenarios = pd.read_csv(scenario_path)

existing_stations = existing_units.drop_duplicates("ID")
print("Recommended sites:", len(recommendations))
print("Existing public stations shown for context:", len(existing_stations))

## Final Network Map

Marker shape distinguishes charging role, marker size represents initial ports, and existing stations are shown as small neutral points.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 11), facecolor="white")
study_area.plot(ax=ax, color="#F5F6F7", edgecolor="#4B5563", linewidth=0.8)
interstates.plot(ax=ax, color="#6B7280", linewidth=1.4, alpha=0.75, label="Interstate")
existing_stations.plot(ax=ax, color="#9CA3AF", marker=".", markersize=12, alpha=0.65, label="Existing public station")

styles = {
    "Level 2": {"color": "#2F6B9A", "marker": "o"},
    "DC fast": {"color": "#D97706", "marker": "D"},
}
for role, style in styles.items():
    subset = recommendations.loc[recommendations["role"] == role]
    if subset.empty:
        continue
    subset.plot(
        ax=ax, color=style["color"], marker=style["marker"],
        markersize=35 + subset["initial_ports"] * 7,
        edgecolor="white", linewidth=0.8, alpha=0.92,
        label=f"Recommended {role}",
    )

for row in recommendations.itertuples():
    ax.annotate(
        row.site_id, (row.geometry.x, row.geometry.y), xytext=(4, 4),
        textcoords="offset points", fontsize=6.5, color="#30343B",
    )

ax.set_title("Recommended EV Charging Network", loc="left", fontsize=19, weight="bold", color="#20242A")
ax.text(0, 1.01, "Planning-level Level 2 and DC-fast sites; marker size represents moderate-scenario initial ports", transform=ax.transAxes, fontsize=10, color="#59616B")
ax.legend(frameon=False, loc="lower left")
ax.set_axis_off()
ax.text(0, -0.025, "Sources: DOE AFDC, 2020–2024 ACS, Census TIGER/Line, GDOT, FEMA, USFWS, USGS, OpenStreetMap contributors.", transform=ax.transAxes, fontsize=8, color="#6B7280")
plt.tight_layout()

final_map = MAP_DIR / "final_recommended_ev_charging_network.png"
fig.savefig(final_map, dpi=300, bbox_inches="tight", facecolor="white")
plt.show()
print("Saved:", final_map)

## Scenario Summary

In [ ]:
scenario_summary = (
    scenarios.merge(recommendations[["site_id", "role"]], on="site_id", validate="many_to_one")
    .groupby(["scenario", "role"], as_index=False)
    .agg(sites=("site_id", "nunique"), ports=("recommended_ports", "sum"))
)

scenario_order = ["current", "moderate", "high"]
scenario_summary["scenario"] = pd.Categorical(scenario_summary["scenario"], scenario_order, ordered=True)
scenario_summary = scenario_summary.sort_values(["scenario", "role"])
display(scenario_summary)

pivot = scenario_summary.pivot(index="scenario", columns="role", values="ports").fillna(0)
pivot = pivot.reindex(scenario_order)
ax = pivot.plot(
    kind="bar", figsize=(9, 5.5), color={"Level 2": "#2F6B9A", "DC fast": "#D97706"},
    edgecolor="white", width=0.7,
)
ax.set_title("Recommended Ports by Adoption Scenario", loc="left", fontsize=15, weight="bold")
ax.set_xlabel("")
ax.set_ylabel("Recommended ports")
ax.tick_params(axis="x", rotation=0)
ax.legend(title="Charging role", frameon=False)
ax.grid(axis="y", color="#E5E7EB", linewidth=0.7)
ax.set_axisbelow(True)
plt.tight_layout()

scenario_figure = FIGURE_DIR / "recommended_ports_by_scenario.png"
plt.savefig(scenario_figure, dpi=300, bbox_inches="tight", facecolor="white")
plt.show()
print("Saved:", scenario_figure)

## Portfolio Recommendation Table

In [ ]:
portfolio_fields = [
    "site_id", "name", "county_name", "role", "power_recommendation",
    "connector_recommendation", "initial_ports", "expansion_ports",
    "estimated_site_kw_initial", "protected_land_review",
]
portfolio_table = recommendations[portfolio_fields].sort_values(["role", "initial_ports"], ascending=[True, False])
portfolio_path = TABLE_DIR / "final_portfolio_recommendations.csv"
portfolio_table.to_csv(portfolio_path, index=False)
display(portfolio_table)
print("Saved:", portfolio_path)

## Validation

In [ ]:
assert recommendations["site_id"].is_unique
assert recommendations["initial_ports"].between(4, 24).all()
assert (recommendations["initial_ports"] <= recommendations["expansion_ports"]).all()
assert set(recommendations["role"]).issubset({"Level 2", "DC fast"})
assert len(scenarios) == len(recommendations) * 3
assert all(path.exists() for path in [final_map, scenario_figure, portfolio_path])

print("All final grain, role, port, scenario, and output checks passed.")
print("Portfolio status: Ready to document after visual inspection of exported figures.")